<a href="https://colab.research.google.com/github/maru415/ECON-499/blob/main/foottraffic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
from getpass import getpass

api_key = getpass("Enter Dewey API Key: ").strip()

print("Key length:", len(api_key))
print("Correct prefix:", api_key.startswith("akv1_"))
import subprocess
import sys

subprocess.run([
    sys.executable, "-m", "deweypy",
    "--api-key", api_key,
    "speedy-download",
    "prj_qz3n4nfy__cdst_ksf3bqzjycrqkdpr",
    "--partition-key-after", "2024-01-01",
    "--partition-key-before", "2024-01-31",
    "--num-workers", "2"
])

Enter Dewey API Key: ··········
Key length: 40
Correct prefix: True


CompletedProcess(args=['/usr/bin/python3', '-m', 'deweypy', '--api-key', 'akv1_HU73wnIKqLBtJewJNm9DyahASUCAGagGxg2', 'speedy-download', 'prj_qz3n4nfy__cdst_ksf3bqzjycrqkdpr', '--partition-key-after', '2024-01-01', '--partition-key-before', '2024-01-31', '--num-workers', '2'], returncode=1)

In [4]:
import requests

url = "https://api.deweydata.io/api/v1/external/data/prj_qz3n4nfy__cdst_ksf3bqzjycrqkdpr"

response = requests.get(
    url,
    headers={"X-API-KEY": api_key},
    timeout=60,
    stream=True
)

print("Status code:", response.status_code)
print("Content type:", response.headers.get("Content-Type"))
print("Response preview:")
print(next(response.iter_content(chunk_size=2000), b"").decode(
    "utf-8", errors="replace"
))

response.close()

Status code: 200
Content type: application/json
Response preview:
{"download_links":[{"link":"https://downloads.deweydata.io/api/v2/downloads/01a1fc48-6f5a-79e1-8d64-27d6146ab237.snappy.parquet?secret=olm4mpZrc8VP1nn","partition_key":"2019-01-01","file_name":"2019-01-01--data_01c88ba0-020d-88f3-0042-fa070cff84c6_209_2_0.snappy.parquet","file_extension":".snappy.parquet","file_size_bytes":290478393,"modified_at":"2026-11-20T21:24:20.180000+00:00","external_id":"dwllv2_01a1fc48-6f5a-79e1-8d64-27d6146ab237"},{"link":"https://downloads.deweydata.io/api/v2/downloads/01a1fc48-6f5a-79e1-8d64-27e9111d7bdc.snappy.parquet?secret=j2UxxlM_j7gLJ6J","partition_key":"2019-02-01","file_name":"2019-02-01--data_01c88ba0-020d-88f3-0042-fa070cff84c6_109_0_0.snappy.parquet","file_extension":".snappy.parquet","file_size_bytes":284534495,"modified_at":"2026-11-20T21:23:49.589000+00:00","external_id":"dwllv2_01a1fc48-6f5a-79e1-8d64-27e9111d7bdc"},{"link":"https://downloads.deweydata.io/api/v2/downloads/01a1fc

In [5]:
import pyarrow.parquet as pq

p = pq.ParquetFile(
    "01a1fc48-6f5a-79e1-8d64-27d6146ab237.snappy.parquet"
)

print("Number of rows:", p.metadata.num_rows)
print("Columns:", p.schema.names)

FileNotFoundError: [Errno 2] Failed to open local file '01a1fc48-6f5a-79e1-8d64-27d6146ab237.snappy.parquet'. Detail: [errno 2] No such file or directory

In [7]:
import glob

paths = glob.glob("/content/**/*.parquet", recursive=True)

print(paths)

[]


In [1]:
from google.colab import files

uploaded = files.upload()

Saving 01a1fc48-6f5a-79e1-8d64-27d6146ab237.snappy.parquet to 01a1fc48-6f5a-79e1-8d64-27d6146ab237.snappy.parquet


In [2]:
import glob
import pyarrow.parquet as pq

paths = glob.glob("/content/**/*.parquet", recursive=True)

if not paths:
    print("No Parquet files found.")
else:
    p = pq.ParquetFile(paths[0])

    print("File:", paths[0])
    print("Number of rows:", p.metadata.num_rows)
    print("Number of row groups:", p.num_row_groups)
    print("\nColumns:")

    for name in p.schema_arrow.names:
        print(name)

File: /content/01a1fc48-6f5a-79e1-8d64-27d6146ab237.snappy.parquet
Number of rows: 7441
Number of row groups: 3

Columns:
AFTERNOON_TEA_DEVICE_HOME_AREAS
AREA
AREA_TYPE
BREAKFAST_DEVICE_HOME_AREAS
DATE_RANGE_END
DATE_RANGE_START
DAY_COUNTS
DEVICE_COUNTS
DEVICE_DAYTIME_AREAS
DEVICE_HOME_AREAS
DEVICE_TYPE
DINNER_DEVICE_HOME_AREAS
DISTANCE_FROM_HOME
DISTANCE_FROM_PRIMARY_DAYTIME_LOCATION
ISO_COUNTRY_CODE
LUNCH_DEVICE_HOME_AREAS
MEDIAN_DWELL
MONTH
NIGHTLIFE_DEVICE_HOME_AREAS
ORIGIN_AREA_TYPE
POPULARITY_BY_EACH_HOUR
POPULARITY_BY_HOUR_FRIDAY
POPULARITY_BY_HOUR_MONDAY
POPULARITY_BY_HOUR_SATURDAY
POPULARITY_BY_HOUR_SUNDAY
POPULARITY_BY_HOUR_THURSDAY
POPULARITY_BY_HOUR_TUESDAY
POPULARITY_BY_HOUR_WEDNESDAY
REGION
STOPS_BY_DAY
STOPS_BY_EACH_HOUR
STOP_COUNTS
TOP_SAME_DAY_BRAND
TOP_SAME_MONTH_BRAND
WEEKDAY_DEVICE_HOME_AREAS
WEEKEND_DEVICE_HOME_AREAS
WORK_BEHAVIOR_DEVICE_HOME_AREAS
WORK_HOURS_DEVICE_HOME_AREAS
YEAR


In [3]:
columns = [
    "AREA",
    "AREA_TYPE",
    "ORIGIN_AREA_TYPE",
    "REGION",
    "YEAR",
    "MONTH",
    "DEVICE_COUNTS",
    "STOP_COUNTS",
    "DEVICE_HOME_AREAS"
]

batch = next(p.iter_batches(
    batch_size=10,
    columns=columns
))

sample = batch.to_pandas()

for col in columns:
    print("\n" + col)
    print(str(sample[col].iloc[0])[:1500])


AREA
130311107022

AREA_TYPE
Census Block Group

ORIGIN_AREA_TYPE
Census Block Group

REGION
GA

YEAR
2019

MONTH
1

DEVICE_COUNTS
18242

STOP_COUNTS
133452

DEVICE_HOME_AREAS
{"130311107022":1301,"130311107012":833,"130311107032":683,"130311107021":589,"130311107013":561,"130311109011":318,"130311107011":308,"130311104052":299,"130311109013":290,"130311105001":280,"130311104014":280,"130311106031":271,"130311106033":262,"130311104071":252,"130311107031":233,"130311103023":233,"130311106041":224,"130311109023":215,"130311104061":205,"130311102021":205,"130311105002":196,"130311104081":196,"130311109022":187,"130311105003":187,"130311108001":187,"130311106042":168,"130311103021":168,"130311106032":168,"130311109021":159,"130311103013":140,"130311104054":131,"130311109012":131,"130311102011":131,"130510107013":121,"130311106022":102,"130299201012":102,"130311106021":102,"130311106024":93,"130311103022":93,"130311103024":84,"131099701002":84,"130311103011":84,"130299201033":84,"130299201

In [4]:
from collections import Counter

coverage = Counter()
fulton_areas = set()
rows = 0

for batch in p.iter_batches(
    batch_size=50000,
    columns=[
        "YEAR", "MONTH", "REGION",
        "AREA", "AREA_TYPE"
    ]
):
    df = batch.to_pandas()
    rows += len(df)

    ga = df[df["REGION"] == "GA"]

    coverage.update(
        zip(ga["YEAR"], ga["MONTH"])
    )

    mask = (
        (ga["YEAR"].astype(str) == "2024")
        & ga["AREA"].astype(str).str.startswith("13121")
        & (ga["AREA_TYPE"] == "Census Block Group")
    )

    fulton_areas.update(
        ga.loc[mask, "AREA"].astype(str)
    )

print("Total rows:", rows)
print("Georgia year/month coverage:")

for (year, month), count in sorted(coverage.items()):
    print(year, month, ":", count)

print(
    "Fulton County CBGs in 2024:",
    len(fulton_areas)
)

Total rows: 7441
Georgia year/month coverage:
2019 1 : 7441
Fulton County CBGs in 2024: 0


In [5]:
import requests
from getpass import getpass

api_key = getpass("Enter your NEW Dewey API Key: ")

url = (
    "https://api.deweydata.io/api/v1/external/data/"
    "prj_qz3n4nfy__cdst_ksf3bqzjycrqkdpr"
)

response = requests.get(
    url,
    headers={"X-API-KEY": api_key},
    timeout=60
)

print("Status:", response.status_code)

if response.ok:
    data = response.json()
    print("Response type:", type(data).__name__)

    if isinstance(data, dict):
        print("Top-level keys:", list(data.keys()))
        for key, value in data.items():
            if isinstance(value, list):
                print(key, "number of items:", len(value))
                print("First item:", str(value[0])[:1000] if value else "empty")
            else:
                print(key, str(value)[:500])
    else:
        print(str(data)[:2000])
else:
    print("Error:", response.text[:1000])

Enter your NEW Dewey API Key: ··········
Status: 200
Response type: dict
Top-level keys: ['download_links', 'page', 'number_of_files_for_page', 'avg_file_size_for_page', 'partition_column', 'total_files', 'total_pages', 'total_size', 'expires_at', 'zip_file']
download_links number of items: 50
First item: {'link': 'https://downloads.deweydata.io/api/v2/downloads/01a20002-80a7-7661-b143-391036aa5167.snappy.parquet?secret=E7mYLnxDrwfFl_z', 'partition_key': '2019-01-01', 'file_name': '2019-01-01--data_01c88ba0-020d-88f3-0042-fa070cff84c6_209_2_0.snappy.parquet', 'file_extension': '.snappy.parquet', 'file_size_bytes': 290478393, 'modified_at': '2026-11-20T21:24:20.180000+00:00', 'external_id': 'dwllv2_01a20002-80a7-7661-b143-391036aa5167'}
page 1
number_of_files_for_page 50
avg_file_size_for_page 327981446.58
partition_column DATE_RANGE_START
total_files 94
total_pages 2
total_size 32681276311
expires_at 2026-11-22T21:28:13.733233+00:00
zip_file None


In [6]:
all_files = []

for page in range(1, int(data["total_pages"]) + 1):
    r = requests.get(
        url,
        headers={"X-API-KEY": api_key},
        params={"page": page},
        timeout=60
    )
    r.raise_for_status()
    all_files.extend(r.json()["download_links"])

all_files = list({
    f["external_id"]: f for f in all_files
}.values())

files_2024 = [
    f for f in all_files
    if str(f["partition_key"]).startswith("2024-")
]

files_2024.sort(key=lambda f: f["partition_key"])

print("Total files found:", len(all_files))
print("2024 files:", len(files_2024))

for f in files_2024:
    print(
        f["partition_key"],
        round(f["file_size_bytes"] / 1e6, 2),
        "MB"
    )

print(
    "Total 2024 size:",
    round(sum(f["file_size_bytes"]
              for f in files_2024) / 1e9, 2),
    "GB"
)

Total files found: 94
2024 files: 12
2024-01-01 277.4 MB
2024-02-01 297.82 MB
2024-03-01 361.18 MB
2024-04-01 383.26 MB
2024-05-01 390.99 MB
2024-06-01 394.71 MB
2024-07-01 411.19 MB
2024-08-01 375.68 MB
2024-09-01 380.26 MB
2024-10-01 404.89 MB
2024-11-01 399.58 MB
2024-12-01 401.93 MB
Total 2024 size: 4.48 GB


In [9]:
import requests
import pandas as pd

malls = {
    "Lenox Square": [
        "3393 Peachtree Rd NE, Atlanta, GA 30326"
    ],
    "Phipps Plaza": [
        "3500 Peachtree Rd NE, Atlanta, GA 30326",
        "3500 Peachtree Road, Atlanta, GA"
    ],
    "Ponce City Market": [
        "675 Ponce de Leon Ave NE, Atlanta, GA 30308"
    ]
}

base = (
    "https://geocoding.geo.census.gov/"
    "geocoder/geographies/"
)

common = {
    "benchmark": "Public_AR_Current",
    "vintage": "Census2020_Current",
    "format": "json"
}

mall_cbgs = {}

def extract_cbg(geo):
    for layer in ["Census Blocks", "2020 Census Blocks"]:
        blocks = geo.get(layer, [])
        if blocks:
            geoid = str(blocks[0]["GEOID"])
            return geoid[:12]
    return None

for mall, addresses in malls.items():
    for address in addresses:
        r = requests.get(
            base + "onelineaddress",
            params={**common, "address": address},
            timeout=60
        )
        r.raise_for_status()

        matches = r.json()["result"]["addressMatches"]

        if matches:
            cbg = extract_cbg(matches[0]["geographies"])

            if cbg:
                mall_cbgs[mall] = cbg
                print(mall, ":", cbg)
                break

    if mall not in mall_cbgs:
        print(mall, ": No address match")

print("\nFinal results:")
print(mall_cbgs)

Lenox Square : 131210096061
Phipps Plaza : 131210100062
Ponce City Market : 131210013021

Final results:
{'Lenox Square': '131210096061', 'Phipps Plaza': '131210100062', 'Ponce City Market': '131210013021'}


In [10]:
import os
import requests
import pyarrow.parquet as pq
import pandas as pd

mall_cbgs = {
    "Lenox Square": "131210096061",
    "Phipps Plaza": "131210100062",
    "Ponce City Market": "131210013021"
}

target_cbgs = set(mall_cbgs.values())

f = next(
    f for f in files_2024
    if f["partition_key"] == "2024-01-01"
)

path = "/content/test_january_2024.parquet"

if not os.path.exists(path):
    with requests.get(
        f["link"],
        stream=True,
        timeout=(30, 180)
    ) as r:
        if not r.ok:
            raise RuntimeError(
                f"Download failed: HTTP {r.status_code}"
            )
        with open(path + ".part", "wb") as out:
            for chunk in r.iter_content(
                chunk_size=4 * 1024 * 1024
            ):
                if chunk:
                    out.write(chunk)
    os.rename(path + ".part", path)

print("Download complete.")

p_jan = pq.ParquetFile(path)

columns = [
    "AREA",
    "YEAR",
    "MONTH",
    "AREA_TYPE",
    "ORIGIN_AREA_TYPE",
    "DEVICE_COUNTS",
    "STOP_COUNTS",
    "DEVICE_HOME_AREAS",
    "MEDIAN_DWELL"
]

selected = []

for batch in p_jan.iter_batches(
    batch_size=10000,
    columns=columns
):
    df = batch.to_pandas()

    mask = (
        df["AREA"].astype(str).isin(target_cbgs)
        & (df["AREA_TYPE"] == "Census Block Group")
        & (df["ORIGIN_AREA_TYPE"] == "Census Block Group")
    )

    selected.append(df.loc[mask])

jan_results = pd.concat(
    selected, ignore_index=True
)

jan_results["Mall"] = (
    jan_results["AREA"].astype(str).map(
        {v: k for k, v in mall_cbgs.items()}
    )
)

display(jan_results[
    [
        "Mall", "AREA", "YEAR", "MONTH",
        "DEVICE_COUNTS", "STOP_COUNTS", "MEDIAN_DWELL"
    ]
])

print("Matched rows:", len(jan_results))
print("Unique destinations:", jan_results["AREA"].nunique())

Download complete.


,Mall,AREA,YEAR,MONTH,DEVICE_COUNTS,STOP_COUNTS,MEDIAN_DWELL
0,Lenox Square,131210096061,2024,1,330235,1083628,10.450000
1,Phipps Plaza,131210100062,2024,1,222577,798883,12.233333
2,Ponce City Market,131210013021,2024,1,180812,384579,10.400000


Matched rows: 3
Unique destinations: 3


In [11]:
import json
from collections.abc import Mapping

for _, row in jan_results.iterrows():
    raw = row["DEVICE_HOME_AREAS"]

    if isinstance(raw, str):
        origins = json.loads(raw)
    elif isinstance(raw, Mapping):
        origins = dict(raw)
    else:
        print(row["Mall"], "Check data format")
        continue

    print("\nMall:", row["Mall"])
    print("Origin CBGs:", len(origins))
    print("Reported origin counts:", sum(origins.values()))


Mall: Lenox Square
Origin CBGs: 2000
Reported origin counts: 151494

Mall: Phipps Plaza
Origin CBGs: 2000
Reported origin counts: 118022

Mall: Ponce City Market
Origin CBGs: 2000
Reported origin counts: 98028


In [13]:
import os
import json
import tempfile
import requests
import pandas as pd
import pyarrow.parquet as pq

from collections.abc import Mapping

mall_cbgs = {
    "Lenox Square": "131210096061",
    "Phipps Plaza": "131210100062",
    "Ponce City Market": "131210013021"
}

cbg_to_mall = {
    v: k for k, v in mall_cbgs.items()
}

target_cbgs = set(mall_cbgs.values())

output_dir = "/content/atlanta_traffic_2024"
os.makedirs(output_dir, exist_ok=True)

columns = [
    "AREA",
    "AREA_TYPE",
    "ORIGIN_AREA_TYPE",
    "YEAR",
    "MONTH",
    "DEVICE_COUNTS",
    "STOP_COUNTS",
    "MEDIAN_DWELL",
    "DISTANCE_FROM_HOME",
    "DEVICE_HOME_AREAS",
    "WEEKDAY_DEVICE_HOME_AREAS",
    "WEEKEND_DEVICE_HOME_AREAS"
]

def parse_origins(raw):
    if raw is None:
        return {}
    if isinstance(raw, str):
        return json.loads(raw) if raw.strip() else {}
    if isinstance(raw, Mapping):
        return dict(raw)
    if isinstance(raw, list):
        return dict(raw)
    if pd.api.types.is_scalar(raw) and pd.isna(raw):
        return {}
    raise TypeError("Unexpected origin data format")

In [14]:
for idx, f in enumerate(files_2024, 1):

    date = f["partition_key"]

    summary_path = (
        f"{output_dir}/summary_{date}.csv"
    )

    flows_path = (
        f"{output_dir}/flows_{date}.csv"
    )

    if (
        os.path.exists(summary_path)
        and os.path.exists(flows_path)
    ):
        print(date, "Already processed")
        continue

    print(f"[{idx}/12] Processing {date}", flush=True)

    with tempfile.TemporaryDirectory() as tmp:

        path = os.path.join(tmp, "monthly.parquet")
        january_path = "/content/test_january_2024.parquet"

        if date == "2024-01-01" and os.path.exists(january_path):
            path = january_path
        else:
            with requests.get(
                f["link"],
                stream=True,
                timeout=(30, 180)
            ) as r:
                if not r.ok:
                    raise RuntimeError(
                        f"Download failed: HTTP {r.status_code}"
                    )
                with open(path, "wb") as out:
                    for chunk in r.iter_content(
                        chunk_size=4 * 1024 * 1024
                    ):
                        if chunk:
                            out.write(chunk)

        parquet = pq.ParquetFile(path)

        summaries = []
        flows = []

        for batch in parquet.iter_batches(
            batch_size=10000,
            columns=columns
        ):

            df = batch.to_pandas()

            selected = df[
                df["AREA"].astype(str).isin(target_cbgs)
                & (df["AREA_TYPE"] == "Census Block Group")
                & (df["ORIGIN_AREA_TYPE"] == "Census Block Group")
            ]

            for _, row in selected.iterrows():

                destination = str(row["AREA"])
                mall = cbg_to_mall[destination]

                origin_data = parse_origins(
                    row["DEVICE_HOME_AREAS"]
                )

                summaries.append({
                    "Mall": mall,
                    "Destination_CBG": destination,
                    "Year": row["YEAR"],
                    "Month": row["MONTH"],
                    "Device_Counts": row["DEVICE_COUNTS"],
                    "Stop_Counts": row["STOP_COUNTS"],
                    "Median_Dwell": row["MEDIAN_DWELL"],
                    "Distance_From_Home": row["DISTANCE_FROM_HOME"],
                    "Reported_Origins": len(origin_data),
                    "Reported_Origin_Counts": sum(
                        origin_data.values()
                    )
                })

                origin_columns = {
                    "All": "DEVICE_HOME_AREAS",
                    "Weekday": "WEEKDAY_DEVICE_HOME_AREAS",
                    "Weekend": "WEEKEND_DEVICE_HOME_AREAS"
                }

                for period, column in origin_columns.items():

                    origins = parse_origins(row[column])

                    for origin, count in origins.items():

                        flows.append({
                            "Mall": mall,
                            "Destination_CBG": destination,
                            "Origin_CBG": str(origin).zfill(12),
                            "Year": row["YEAR"],
                            "Month": row["MONTH"],
                            "Period": period,
                            "Device_Count": count
                        })

        summary = pd.DataFrame(summaries)
        flow_df = pd.DataFrame(flows)

        if len(summary) != 3 or summary["Destination_CBG"].nunique() != 3:
            print("Warning: unexpected destination coverage")
            print(date, "Matched rows:", len(summary))
            continue

        summary.to_csv(summary_path, index=False)
        flow_df.to_csv(flows_path, index=False)

        print(
            date,
            "| Destinations:", len(summary),
            "| Flow rows:", len(flow_df),
            flush=True
        )

print("Processing finished.")

[1/12] Processing 2024-01-01
2024-01-01 | Destinations: 3 | Flow rows: 18000
[2/12] Processing 2024-02-01
2024-02-01 | Destinations: 3 | Flow rows: 18000
[3/12] Processing 2024-03-01
2024-03-01 | Destinations: 3 | Flow rows: 18000
[4/12] Processing 2024-04-01
2024-04-01 | Destinations: 3 | Flow rows: 18000
[5/12] Processing 2024-05-01
2024-05-01 | Destinations: 3 | Flow rows: 18000
[6/12] Processing 2024-06-01
2024-06-01 | Destinations: 3 | Flow rows: 18000
[7/12] Processing 2024-07-01
2024-07-01 | Destinations: 3 | Flow rows: 18000
[8/12] Processing 2024-08-01
2024-08-01 | Destinations: 3 | Flow rows: 18000
[9/12] Processing 2024-09-01
2024-09-01 | Destinations: 3 | Flow rows: 18000
[10/12] Processing 2024-10-01
2024-10-01 | Destinations: 3 | Flow rows: 18000
[11/12] Processing 2024-11-01
2024-11-01 | Destinations: 3 | Flow rows: 18000
[12/12] Processing 2024-12-01
2024-12-01 | Destinations: 3 | Flow rows: 18000
Processing finished.


In [15]:
import glob
import shutil
from google.colab import files

summary_files = sorted(
    glob.glob(f"{output_dir}/summary_*.csv")
)

flow_files = sorted(
    glob.glob(f"{output_dir}/flows_*.csv")
)

summary_all = pd.concat(
    [pd.read_csv(
        f, dtype={"Destination_CBG": str}
    ) for f in summary_files],
    ignore_index=True
)

flows_all = pd.concat(
    [pd.read_csv(
        f,
        dtype={
            "Destination_CBG": str,
            "Origin_CBG": str
        }
    ) for f in flow_files],
    ignore_index=True
)

print("Months processed:", len(summary_files))
print("Summary rows:", len(summary_all))
print("Origin-Destination rows:", len(flows_all))
print("Unique origin CBGs:", flows_all["Origin_CBG"].nunique())

final_dir = "/content/atlanta_final_data"
os.makedirs(final_dir, exist_ok=True)

summary_all.to_csv(
    f"{final_dir}/Mall_Monthly_Summary.csv",
    index=False
)

flows_all.to_csv(
    f"{final_dir}/Origin_Destination_Flows.csv",
    index=False
)

zip_path = shutil.make_archive(
    "/content/Atlanta_Foot_Traffic_2024",
    "zip",
    final_dir
)

files.download(zip_path)

Months processed: 12
Summary rows: 36
Origin-Destination rows: 216000
Unique origin CBGs: 13907


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>